# Shape Research inspection
Read-only actual data and frozen window checks. No model training or returns.
Scope: observed historical common equity IDs; missingness remains explicit. Fresh OOS has no full windows yet.


In [1]:
from pathlib import Path
import sys, json
root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists():
    if root.parent == root:
        raise RuntimeError("Open this notebook inside stock-radar")
    root = root.parent
sys.path.insert(0, str(root / "src"))
from radar.pit.shape import ShapeResearchDatabase, tensor, canonical_hash
score = json.loads((root / "reports/evidence/shape-research-scorecard-2026-10-08.json").read_text(encoding="utf-8"))
print(json.dumps({"state":score["shape_research_state"], "price":score["price"], "source_locks":score["source_locks_unchanged"]}, indent=2))

{
  "state": "SHAPE_RESEARCH_READY",
  "price": {
    "total_observed_common_member_days": 6929009,
    "available": {
      "numerator": 5796676,
      "denominator": 6929009,
      "pct": 83.6581
    },
    "ready": {
      "numerator": 5775762,
      "denominator": 6929009,
      "pct": 83.3562
    },
    "ready_of_available": {
      "numerator": 5775762,
      "denominator": 5796676,
      "pct": 99.6392
    },
    "default_supported_ready": {
      "numerator": 5372282,
      "denominator": 6929009,
      "pct": 77.5332
    },
    "status": {
      "MISSING": {
        "numerator": 1132333,
        "denominator": 6929009,
        "pct": 16.3419
      },
      "READY_WITH_MINOR_UNCERTAINTY": {
        "numerator": 1681005,
        "denominator": 6929009,
        "pct": 24.2604
      },
      "READY": {
        "numerator": 4094757,
        "denominator": 6929009,
        "pct": 59.0959
      },
      "QUARANTINED": {
        "numerator": 20914,
        "denominator": 6929009,
    

In [2]:
print(json.dumps({"windows": score["windows"], "survivorship": score["survivorship"], "unknown_sensitivity": score["unknown_sensitivity"]}, indent=2))

{
  "windows": [
    {
      "length": 20,
      "split_assignment": "fresh_oos",
      "including_unknown": 0,
      "supported": 0
    },
    {
      "length": 20,
      "split_assignment": "test",
      "including_unknown": 1084728,
      "supported": 1079929
    },
    {
      "length": 20,
      "split_assignment": "train",
      "including_unknown": 3015847,
      "supported": 2705603
    },
    {
      "length": 20,
      "split_assignment": "validation",
      "including_unknown": 1072511,
      "supported": 999559
    },
    {
      "length": 40,
      "split_assignment": "fresh_oos",
      "including_unknown": 0,
      "supported": 0
    },
    {
      "length": 40,
      "split_assignment": "test",
      "including_unknown": 966275,
      "supported": 961925
    },
    {
      "length": 40,
      "split_assignment": "train",
      "including_unknown": 2883295,
      "supported": 2578979
    },
    {
      "length": 40,
      "split_assignment": "validation",
      "including

In [3]:
with ShapeResearchDatabase.current(root) as db:
    w = db.window("SEC-0000886158-COMMON", "2023-02-03", 126)
    assert w.ohlcv.date.max().strftime("%Y-%m-%d") == "2023-02-03"
    assert canonical_hash(w.ohlcv) == w.metadata["ohlcv_hash"]
    assert w.metadata["split_assignment"] == "train"
    print(json.dumps(w.metadata, indent=2, default=str))
    print("tensor shape:", tensor(w.normalized).shape)
    print("training input shape:", w.training_tensor().shape)


{
  "security_id": "SEC-0000886158-COMMON",
  "series_id": "SEC-0000886158-COMMON|Alpaca SIP|raw",
  "source": "Alpaca SIP",
  "basis": "raw",
  "priority": 2,
  "historical_ticker": "BBBY",
  "membership_status": "confirmed_member",
  "shape_research_status": "READY",
  "decision_date": "2023-02-03 00:00:00",
  "length": 126,
  "window_start": "2022-08-05 00:00:00",
  "start_session": 234,
  "start_run": 1.0,
  "start_ready": true,
  "supported_membership": true,
  "session_no": 359,
  "run_no": 1.0,
  "shape_research_ready": true,
  "split_assignment": "train",
  "window_end": "2023-02-03 00:00:00",
  "ohlcv_hash": "fde0520238ad84e28ab51d43aeb16f1442697e206ebb2486c1fd295f8584971d",
  "normalized_hash": "4c5d87a4de16d2b03c5e84e9e95b4a97f3b4e8ab1fc6f4df3804fe20b4faa524",
  "shape_readiness_status": "READY_WITH_MINOR_UNCERTAINTY",
  "adjustment_mode": "raw",
  "source_hash": "86ff0ef43f339b0566e86fe52358b648038ac53f4c82f0b453e231d8b141738c",
  "dataset_version": "0ee4ddb99d115c467e75967